In [1]:
# utils
import pprint
import os

# policy
from huggingface_hub import snapshot_download
from lerobot.configs.policies import PreTrainedConfig

# lerobot
from lerobot.record import RecordConfig, DatasetRecordConfig
from lerobot.constants import OBS_ENV_STATE, ACTION, OBS_STATE, OBS_IMAGES

# torch
from torch import cuda

# my code
from robot.robot_config import robot_config, robot_ext_config
from robot.robot_const import FOLDED_START_POSE, JOINT_CURRENT_NAMES, JOINT_POS_NAMES
from src.paths import REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR
from src.record_extended import record_extended
from src.utils import check_resume

# yolo
from src.yolo.yolo_lerobot_processor import YoloAnnotateProcessorStep
from src.yolo.yolo_policy_preprocessor import FilterEnvObsProcessorStep, RemoveFeatureProcessorStep  # noqa: F401

# set up env secrets
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

/home/jonathan/miniforge3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


#### 1. Set Evaluation Params:

In [2]:
# global settings
PUSH_TO_HUB      = False
SAVE_TO_DATASET  = False
NUM_EPISODES     = 8
FPS              = 30
EPISODE_TIME_SEC = 60
RESET_TIME_SEC   = 5

# ACT policy settings
ACT_REPO_NAME       = 'so101_pick_pen-bbox'
ACT_EXPERIMENT_NAME = 'bbox_v0'
POLICY_CHECKPOINT   = 'last'
POLICY_TYPE         = 'act'
EVAL_TYPE           = 'real_v0'

# special features
REMOVE_FEATURES = None 
USE_EXT_ROBOT   = False

# YOLO model settings
YOLO_REPO_NAME        = 'so101_pick_pen'
YOLO_EXPERIMENT_NAME  = 'v0'
YOLO_INCLUDE_ROTATION = False                       # should I use x,y,r for OBB or just x,y 

Log to HF

In [3]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")

#### 2. Initialize Policy and Overrides:

In [4]:
# Resolve path or HF id
if POLICY_CHECKPOINT is None:
    POLICY_CHECKPOINT = "last"
policy_path = POLICIES_DIR / POLICY_TYPE / ACT_REPO_NAME / ACT_EXPERIMENT_NAME / "checkpoints" / POLICY_CHECKPOINT / "pretrained_model"

if not policy_path.exists():
    print('Loading policy from HF')
    policy_id = f"{HF_NAME}/{POLICY_TYPE}-{ACT_REPO_NAME}-{ACT_EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = policy_id,
    revision               = "main",
    local_dir              = str(policy_path),
    local_dir_use_symlinks = False
    )

policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path # type: ignore
pprint.pprint(policy_config.input_features)

{'observation.environment_state': PolicyFeature(type=<FeatureType.ENV: 'ENV'>,
                                                shape=(4,)),
 'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                             shape=(3, 480, 640)),
 'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                               shape=(3, 480, 640)),
 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>,
                                    shape=(6,))}


Policy Overrides:

In [5]:
print(f"Original n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")
# policy_config.n_action_steps = 100
# policy_config.temporal_ensemble_coeff = 0.01
print(f"Override n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")

Original n_action_steps = 100, temporal_ensemble = None
Override n_action_steps = 100, temporal_ensemble = None


Policy processor prarams:

In [6]:
if REMOVE_FEATURES is not None:
    policy_config.policy_preprocessor_config = {"remove_feature_processor": {"remove_feature_names": REMOVE_FEATURES}}
else:
    policy_config.policy_preprocessor_config = None

### 3. Build YOLO Model
Fetch model:

In [7]:
yolo_model_path = POLICIES_DIR / 'yolo' / YOLO_REPO_NAME / YOLO_EXPERIMENT_NAME / 'best.pt'
if not yolo_model_path.exists():
    print('Loading yolo model from HF')
    model_id = f"{HF_NAME}/yolov8s-obb-{YOLO_REPO_NAME}-{YOLO_EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = model_id,
    revision               = "main",
    local_dir              = str(yolo_model_path.parent),
    local_dir_use_symlinks = False
    )

# build processor
yolo_processor = YoloAnnotateProcessorStep(model_path = yolo_model_path, cam_name = 'top_cam', xy_only = not YOLO_INCLUDE_ROTATION)

#### 4. Build Evaluation Dataset
Used for data storage of the evaluation dataset

In [8]:
dataset_path = EVAL_DIR / POLICY_TYPE / ACT_REPO_NAME / f"{ACT_EXPERIMENT_NAME}-{EVAL_TYPE}"
resume = check_resume(dataset_path)

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{ACT_REPO_NAME}_{ACT_EXPERIMENT_NAME}-{EVAL_TYPE}",
    single_task                         = f"eval dataset for {ACT_REPO_NAME} with policy {ACT_EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_ext_config if USE_EXT_ROBOT else robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

In [9]:
policy_config.input_features

{'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(6,)),
 'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 480, 640)),
 'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 480, 640)),
 'observation.environment_state': PolicyFeature(type=<FeatureType.ENV: 'ENV'>, shape=(4,))}

### 5. Configure datset ovverrides
If need to specify additional featuers (such as env state or other), or override the default. If none it will default according to the robot config. 
Uncomment if needed:

In [10]:
if USE_EXT_ROBOT:
    obs_names = JOINT_POS_NAMES + JOINT_CURRENT_NAMES
else:
    obs_names = JOINT_POS_NAMES

ds_features_override = {
ACTION: {
    'dtype': 'float32', 
    'names': JOINT_POS_NAMES,
    'shape': (6,)
    },
OBS_STATE: {
    'dtype': 'float32', 
    'names': obs_names,
    'shape': (len(obs_names),)
    }, 
f"{OBS_IMAGES}.wrist_cam": {
    "dtype": "video",
    "shape": (480, 640, 3),
    "names": ["height", "width", "channels"],
    },
f"{OBS_IMAGES}.top_cam": {
    "dtype": "video",
    "shape": (480, 640, 3),
    "names": ["height", "width", "channels"],
    },
}

# add the environment variable
names = (
    ["source_x", "source_y", "source_r", "target_x", "target_y", "target_r"]
    if YOLO_INCLUDE_ROTATION
    else ["source_x", "source_y", "target_x", "target_y"]
)

ds_features_override[OBS_ENV_STATE] = {
    "dtype": "float32",
    "names": names,
    "shape": (len(names),),
}

#### 6. Run inference

In [ ]:
rc.resume = check_resume(dataset_path)
record_extended(
    rc, 
    extra_robot_processor = [yolo_processor],
    extra_features        = locals().get("new_feature_list"),
    ds_features_override  = locals().get("ds_features_override"),
    save_to_ds            = SAVE_TO_DATASET,
    reset_pose            = FOLDED_START_POSE,
    give_feedback         = False,
    log_to_file           = False,
    replace_image_key     = "observation.images.top_cam",
    replace_with_key      = "top_cam_bbox",
)

Loading weights from local directory


INFO 2026-01-04 23:03:54 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-Sonix_Technology_Co.__Ltd._USB2.0_CAM1_USB2.0_CAM1-video-index0) connected.
INFO 2026-01-04 23:03:57 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-046d_Logitech_BRIO_8F54E371-video-index0) connected.
INFO 2026-01-04 23:03:57 follower.py:104 so_101_follower SO101Follower connected.
INFO 2026-01-04 23:03:58 ls/utils.py:227 Recording episode 0
[2026-01-04T21:03:58Z INFO  winit::platform_impl::linux::x11::window] Guessed window scale factor: 1
[2026-01-04T21:03:58Z WARN  wgpu_hal::gles::egl] No config found!
[2026-01-04T21:03:58Z WARN  wgpu_hal::gles::egl] EGL says it can present to the window but not natively
[2026-01-04T21:03:58Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2026-01-04T21:03:58Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2026-01-04T21:03:58Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2

Left arrow key pressed. Exiting loop and rerecord the last episode...


INFO 2026-01-04 23:06:32 ls/utils.py:227 Re-record episode
INFO 2026-01-04 23:06:32 ls/utils.py:227 Recording episode 0
INFO 2026-01-04 23:07:24 ls/utils.py:227 Reset the environment


Left arrow key pressed. Exiting loop and rerecord the last episode...


...

Left arrow key pressed. Exiting loop and rerecord the last episode...


INFO 2026-01-04 23:08:13 ls/utils.py:227 Re-record episode
INFO 2026-01-04 23:08:13 ls/utils.py:227 Recording episode 0
INFO 2026-01-04 23:09:13 ls/utils.py:227 Reset the environment
INFO 2026-01-04 23:09:14 ls/utils.py:227 Recording episode 0
[2026-01-04T21:09:25Z WARN  re_sdk_comms::server] Closing connection to client at 127.0.0.1:44576: The receiving end of the channel was closed
[2026-01-04T21:09:25Z WARN  re_sdk_comms::server] Closing connection to client at 127.0.0.1:33882: The receiving end of the channel was closed
[2026-01-04T21:09:25Z WARN  re_sdk_comms::server] Closing connection to client at 127.0.0.1:33888: The receiving end of the channel was closed
[2026-01-04T21:09:26Z WARN  re_sdk_comms::buffered_client] Failed to send message after 3 attempts: Failed to send to Rerun server at 127.0.0.1:9876: Connection reset by peer (os error 104)


KeyboardInterrupt: 

[2026-01-04T21:09:28Z WARN  re_sdk_comms::buffered_client] Dropping messages because tcp client has timed out.
[2026-01-04T21:09:28Z WARN  re_sdk_comms::buffered_client] Dropping messages because tcp client has timed out.


Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Right arrow key pressed. Exiting loop...
